# BEI Pipeline v5 — Full Rebuild
**Split strategy**: Grouped random (ticker+week group), no leakage
**Data range**: 2020–2026 full (train+val+test dari range yang sama)

In [ ]:
import re, json, logging, warnings, hashlib
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("BEIPipeline")

CAPITAL_STOPWORDS = {
    "PADA","DARI","IHSG","RUPS","BUMN","ESDM","APBN","APBD","RUPST","POJK",
    "OJK","IDX","BEI","KSEI","YANG","DENGAN","UNTUK","DALAM","ATAU","JUGA",
    "AKAN","SUDAH","OLEH","ATAS","AGAR","BAGI","SAAT","SAJA","HARGA","TAHUN",
    "BULAN","PASAR","MODAL","SAHAM","EMITEN","DIREKSI","KOMISARIS","RUPIAH",
    "DIVIDEN","BURSA","BANK","DANA","BELI","JUAL","NAIK","TURUN","LABA",
    "RUGI","KURS","NILAI","RATA","TOTAL","HASIL",
}

ANOMALY_DISTRIBUTION  = "DISTRIBUSI/DUMP"
ANOMALY_ACCUMULATION  = "AKUMULASI/PULLBACK"
SIGNAL_VALID_CATALYST = "KATALIS_VALID"
SIGNAL_NEUTRAL        = "NETRAL"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

BASE         = "/content/drive/MyDrive/finances/Finance/filter_news/claude"
MASTER_PATH  = f"{BASE}/master_dataset.parquet"
MACRO_NEWS   = f"{BASE}/data/news/macro_news.json"
MARKET_NEWS  = f"{BASE}/data/news/market_news.json"
SECTORS_PATH = f"{BASE}/data/stocks_sectors.json"
SIGNALS_PATH = f"{BASE}/signals_v5.json"
DATASET_PATH = f"{BASE}/training_dataset_v5.parquet"


In [ ]:
class SentimentEngine:
    """Raw PyTorch inference — 5-10x lebih cepat dari pipeline()."""
    MODEL_NAME = "w11wo/indonesian-roberta-base-sentiment-classifier"
    LABEL_MAP  = {
        "positive":"Positif","negative":"Negatif","neutral":"Netral",
        "LABEL_0":"Positif","LABEL_1":"Negatif","LABEL_2":"Netral",
        "Positive":"Positif","Negative":"Negatif","Neutral":"Netral",
    }
    POS_KW = {"naik","menguat","melonjak","tumbuh","profit","laba","untung",
               "meningkat","rekor","ekspansi","dividen","positif","rally",
               "bullish","buy","surplus","pertumbuhan","gain","lonjakan",
               "terbang","hijau","cerah","optimis","outperform"}
    NEG_KW = {"turun","melemah","anjlok","jatuh","rugi","kerugian","susut",
               "merosot","gagal","pailit","defisit","negatif","bearish","sell",
               "bangkrut","koreksi","tekanan","resesi","krisis","loss","ambles",
               "terpuruk","merah","longsor","rontok","jeblok"}

    def __init__(self, batch_size=128, max_length=128, neutral_threshold=0.70):
        self.batch_size = batch_size
        self.max_length = max_length
        self.neutral_threshold = neutral_threshold
        self.tokenizer = self.model = None
        self.id2label  = {}

    def _load(self):
        if self.model: return
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        print(f"⏳ Loading {self.MODEL_NAME}...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.MODEL_NAME)
        self.model = AutoModelForSequenceClassification.from_pretrained(self.MODEL_NAME).to(DEVICE)
        self.model.eval()
        self.id2label = {int(k): self.LABEL_MAP.get(v,v)
                         for k,v in self.model.config.id2label.items()}
        print(f"✅ Loaded | Labels: {self.id2label}")

    def _truncate(self, t): return " ".join(str(t).split()[:200])

    def _keyword_override(self, text):
        w = set(text.lower().split())
        pos = len(w & self.POS_KW); neg = len(w & self.NEG_KW)
        if pos == 0 and neg == 0: return None
        return "Positif" if pos > neg else ("Negatif" if neg > pos else None)

    @torch.no_grad()
    def predict_batch(self, texts: List[str]) -> List[Tuple[str, float]]:
        self._load()
        truncated = [self._truncate(t) for t in texts]
        results   = []
        for i in tqdm(range(0, len(truncated), self.batch_size), desc="🚀 GPU Inference"):
            batch  = truncated[i:i+self.batch_size]
            inputs = self.tokenizer(batch, padding=True, truncation=True,
                                    max_length=self.max_length, return_tensors="pt").to(DEVICE)
            try:
                probs   = torch.softmax(self.model(**inputs).logits, dim=-1)
                top_ids = probs.argmax(dim=-1).cpu().tolist()
                top_scr = probs.max(dim=-1).values.cpu().tolist()
                for j, (lid, sc) in enumerate(zip(top_ids, top_scr)):
                    label = self.id2label.get(lid, "Netral")
                    sc    = round(sc, 4)
                    if label == "Netral" and sc > self.neutral_threshold:
                        override = self._keyword_override(batch[j])
                        if override: label, sc = override, round(sc*0.75, 4)
                    results.append((label, sc))
            except RuntimeError as e:
                if "out of memory" not in str(e).lower(): raise
                torch.cuda.empty_cache()
                for text in batch:
                    inp = self.tokenizer([text], padding=True, truncation=True,
                                         max_length=self.max_length, return_tensors="pt")
                    p   = torch.softmax(self.model(**inp).logits, dim=-1)
                    lid = p.argmax().item()
                    results.append((self.id2label.get(lid,"Netral"), round(p.max().item(),4)))
        return results

print("✅ SentimentEngine ready")


In [ ]:
class TickerExtractor:
    TICKER_PATTERN  = re.compile(r"\b([A-Z]{4})\b")
    CONTEXT_SIGNALS = {
        "saham","emiten","tbk","pt","kode","harga","beli","jual","naik","turun",
        "rally","koreksi","listing","ipo","rights","buyback","dividen","laba",
        "rugi","kinerja","laporan","manajemen","direktur",
    }
    def __init__(self, valid_tickers: set, require_context=True):
        self.valid_tickers  = valid_tickers
        self.require_context = require_context

    def _has_context(self, text, ticker):
        words = text.lower().split(); tl = ticker.lower()
        for i,w in enumerate(words):
            if w == tl or tl in w:
                if any(kw in words[max(0,i-5):i+6] for kw in self.CONTEXT_SIGNALS):
                    return True
        return False

    def extract(self, title, content):
        title_cands   = self.TICKER_PATTERN.findall(title or "")
        content_cands = self.TICKER_PATTERN.findall(content or "")
        combined      = (title or "") + " " + (content or "")
        seen, valid   = set(), []
        for c in title_cands:
            if c not in CAPITAL_STOPWORDS and c in self.valid_tickers and c not in seen:
                valid.append(c); seen.add(c)
        for c in content_cands:
            if c in CAPITAL_STOPWORDS or c not in self.valid_tickers or c in seen: continue
            if not self.require_context or self._has_context(combined, c):
                valid.append(c); seen.add(c)
        return valid


class AnomalyDetector:
    def classify(self, sentiment, open_price, close_price, volume, avg_volume):
        if pd.isna(open_price) or open_price == 0:
            return {"anomaly_type":SIGNAL_NEUTRAL, "price_change_pct":0.0, "volume_ratio":1.0}
        pct  = ((close_price - open_price) / open_price) * 100
        vrat = volume / avg_volume if avg_volume and avg_volume > 0 else 1.0
        up   = close_price > open_price
        if   sentiment=="Positif" and up:     atype = SIGNAL_VALID_CATALYST
        elif sentiment=="Positif" and not up: atype = ANOMALY_DISTRIBUTION
        elif sentiment=="Negatif" and up:     atype = ANOMALY_ACCUMULATION
        else:                                 atype = SIGNAL_NEUTRAL
        return {"anomaly_type":atype, "price_change_pct":round(pct,4), "volume_ratio":round(vrat,4)}


class BandarmologiConfirmation:
    def confirm(self, anomaly_type, net_foreign, net_nonfin):
        if anomaly_type not in (ANOMALY_DISTRIBUTION, ANOMALY_ACCUMULATION):
            return {"bdm_status":"N/A","bdm_reason":"Bukan anomali"}
        if pd.isna(net_foreign) and pd.isna(net_nonfin):
            return {"bdm_status":"NO_DATA","bdm_reason":"Data BDM tidak tersedia"}
        nf = net_foreign if not pd.isna(net_foreign) else 0.0
        if anomaly_type == ANOMALY_DISTRIBUTION:
            return ({"bdm_status":"CONFIRMED","bdm_reason":f"Foreign JUAL {nf:+,.0f}"}
                    if nf < 0 else {"bdm_status":"UNCONFIRMED","bdm_reason":f"Foreign BELI {nf:+,.0f}"})
        return ({"bdm_status":"CONFIRMED","bdm_reason":f"Akumulasi foreign={nf:+,.0f}"}
                if nf > 0 else {"bdm_status":"UNCONFIRMED","bdm_reason":"Tidak ada pembelian institusional"})


class MacroContextEvaluator:
    def evaluate(self, sektor, usdidr, brent_usd, coal_usd, bi_rate=None):
        r = {"macro_tailwind":[], "macro_headwind":[]}
        if any(pd.isna(v) for v in [usdidr, brent_usd, coal_usd]): return r
        s = str(sektor or "")
        if any(x in s for x in ["Energi","Batu Bara","Minyak"]):
            (r["macro_tailwind"] if coal_usd>100 else r["macro_headwind"]).append(f"Coal ${coal_usd:.0f}/t")
            (r["macro_tailwind"] if brent_usd>70  else r["macro_headwind"]).append(f"Brent ${brent_usd:.0f}/bbl")
        if any(x in s for x in ["Konsumen","Teknologi","Properti","Transportasi"]):
            (r["macro_headwind"] if usdidr>16000 else r["macro_tailwind"]).append(f"USD/IDR {usdidr:,.0f}")
        if bi_rate and any(x in s for x in ["Perbankan","Keuangan"]):
            (r["macro_headwind"] if bi_rate>6.0 else r["macro_tailwind"]).append(f"BI {bi_rate:.2f}%")
        return r

print("✅ TickerExtractor, AnomalyDetector, BDMConfirmation, MacroEvaluator ready")


In [ ]:
class BEIPipeline:
    """
    Pipeline v5 — identik dengan v4 tapi menambahkan:

    GROUP_ID untuk grouped-random split:
    ─────────────────────────────────────
    Setiap row di training_dataset mendapat group_id berupa hash dari
    (ticker, week_of_year). Semua sinyal untuk ticker yang sama dalam
    minggu yang sama SELALU masuk ke split yang sama (train/val/test).

    Ini mencegah leakage:
    - Berita ticker BBRI tanggal 15 Jan → group = "BBRI_2022W02"
    - Berita ticker BBRI tanggal 17 Jan → group = "BBRI_2022W02" (SAMA)
    - Keduanya PASTI masuk ke split yang sama, tidak bisa satu di train
      dan satu di test
    """

    def __init__(self, master_df, news_df, sector_df=None,
                 batch_size=128, max_length=128, neutral_threshold=0.70):

        self.master_df = master_df.copy()
        self.master_df["date"] = pd.to_datetime(self.master_df["date"])

        ticker_col = self._find_col(master_df, ["ticker","kode","symbol"])
        sektor_col = self._find_col(master_df, ["Sektor","sektor","sector"])

        # Sector map langsung dari master_df
        if ticker_col and sektor_col:
            self.sector_map = (
                master_df[[ticker_col, sektor_col]].dropna()
                .drop_duplicates(ticker_col).set_index(ticker_col)[sektor_col].to_dict()
            )
        else:
            self.sector_map = self._build_sector_map(sector_df)
        logger.info(f"Sector map: {len(self.sector_map):,} ticker")

        self.news_df = self._preprocess_news(news_df)

        valid_tickers = set(master_df[ticker_col].dropna().unique()) - {"IHSG",""}
        self.ticker_extractor = TickerExtractor(valid_tickers, require_context=True)
        self.sentiment_engine = SentimentEngine(batch_size, max_length, neutral_threshold)
        self.anomaly_detector = AnomalyDetector()
        self.bdm_confirmation = BandarmologiConfirmation()
        self.macro_evaluator  = MacroContextEvaluator()

        logger.info("📦 Building lookups...")
        self._lookup, self._macro_lookup, self._col = self._build_lookups(master_df, ticker_col)
        logger.info(f"OHLCV lookup: {len(self._lookup):,} | Macro: {len(self._macro_lookup):,}")

    @staticmethod
    def _find_col(df, options):
        cols_l = {c.lower():c for c in df.columns}
        for o in options:
            if o.lower() in cols_l: return cols_l[o.lower()]
        return None

    def _build_sector_map(self, sector_df):
        if sector_df is None or sector_df.empty: return {}
        t = self._find_col(sector_df, ["KodeEmiten","ticker","kode"])
        s = self._find_col(sector_df, ["Sektor","sektor","sector"])
        if not t or not s: return {}
        return dict(zip(sector_df[t].str.upper(), sector_df[s]))

    def _preprocess_news(self, df):
        df = df.copy()
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.dropna(subset=["date"])
        title   = df.get("title", pd.Series([""]*len(df))).fillna("")
        content = df.get("full_content", df.get("summary", pd.Series([""]*len(df)))).fillna("")
        df["text_for_nlp"] = title + " " + content
        before = len(df)
        if "title" in df.columns:
            df = df.drop_duplicates(subset=["date","title"]).reset_index(drop=True)
        logger.info(f"News: {before:,} → {len(df):,} (dedup -{before-len(df):,})")
        return df.sort_values("date").reset_index(drop=True)

    def _build_lookups(self, master_df, ticker_col):
        df = master_df.copy()
        def fc(*opts): return self._find_col(df, list(opts))
        open_c   = fc("open","Open")
        close_c  = fc("close","Close","adj_close")
        vol_c    = fc("volume","Volume")
        hi_c     = fc("high","High")
        lo_c     = fc("low","Low")
        ret_c    = fc("daily_return","return")
        intra_c  = fc("intraday_move","intraday")
        nf_c     = fc("net_foreign","foreign_net")
        nfi_c    = fc("net_nonfin")
        mktcap_c = fc("market_cap","mktcap")
        usd_c    = fc("usdidr","usd_idr")
        brent_c  = fc("brent_usd","brent")
        coal_c   = fc("coal_usd","coal")
        bir_c    = fc("bi_rate_pct","bi_rate")

        if vol_c:
            df["__avg_vol_20"] = df.groupby(ticker_col)[vol_c].transform(
                lambda x: x.rolling(20, min_periods=1).mean())
        if open_c and close_c:
            bad = (df[open_c]==0) & (df[close_c]==0)
            if bad.sum() > 0:
                logger.warning(f"Skip {bad.sum():,} rows (open=close=0)")
                df = df[~bad]

        df["__date_str"] = df["date"].dt.strftime("%Y-%m-%d")
        df["__key"]      = df[ticker_col].astype(str) + "_" + df["__date_str"]

        col = {
            "open":open_c,"close":close_c,"volume":vol_c,"high":hi_c,"low":lo_c,
            "avg_vol":"__avg_vol_20","daily_return":ret_c,"intraday_move":intra_c,
            "net_foreign":nf_c,"net_nonfin":nfi_c,"market_cap":mktcap_c,
            "usdidr":usd_c,"brent":brent_c,"coal":coal_c,"bi_rate":bir_c,
        }
        keep = [c for c in col.values() if c]
        ohlcv_lookup = df.set_index("__key")[keep].to_dict("index")
        macro_cols   = [c for c in [usd_c,brent_c,coal_c,bir_c] if c]
        macro_lookup = df.groupby("__date_str")[macro_cols].first().to_dict("index") if macro_cols else {}
        return ohlcv_lookup, macro_lookup, col

    def _get_ohlcv(self, ticker, date_str):
        for delta in [0,1,2,3]:
            d = (pd.Timestamp(date_str)+pd.Timedelta(days=delta)).strftime("%Y-%m-%d") if delta>0 else date_str
            key = f"{ticker}_{d}"
            if key in self._lookup:
                return self._lookup[key], d
        return None, None

    @staticmethod
    def _make_group_id(ticker: str, date_str: str) -> str:
        """
        Group ID untuk grouped-random split.
        Format: TICKER_YYYYWww  (contoh: BBRI_2022W03)

        Semua berita untuk ticker yang sama dalam minggu kalender yang sama
        mendapat group_id yang identik → PASTI masuk split yang sama.
        """
        try:
            dt      = pd.Timestamp(date_str)
            week_id = f"{dt.year}W{dt.isocalendar()[1]:02d}"
        except Exception:
            week_id = "UNKNOWN"
        return f"{ticker}_{week_id}"

    def run(self, output_path=None):
        output_path = output_path or SIGNALS_PATH
        logger.info(f"BEIPipeline.run() | {len(self.news_df):,} berita")

        # STEP 1: Sentiment
        texts      = self.news_df["text_for_nlp"].astype(str).tolist()
        sent_pairs = self.sentiment_engine.predict_batch(texts)
        self.news_df["sentiment_label"] = [p[0] for p in sent_pairs]
        self.news_df["sentiment_score"] = [p[1] for p in sent_pairs]
        dist = self.news_df["sentiment_label"].value_counts()
        logger.info(f"Sentimen: {dist.to_dict()}")

        # STEP 2: Core loop
        col        = self._col
        all_dates  = self.news_df["date"].dt.strftime("%Y-%m-%d").tolist()
        all_titles = self.news_df.get("title", pd.Series([""]*len(self.news_df))).tolist()
        all_texts  = self.news_df["text_for_nlp"].tolist()
        all_sents  = self.news_df["sentiment_label"].tolist()
        all_scores = self.news_df["sentiment_score"].tolist()

        signals = []
        miss    = 0

        for i in tqdm(range(len(self.news_df)), desc="⚡ Processing"):
            date_str  = all_dates[i]
            sentiment = all_sents[i]
            score     = all_scores[i]
            title     = all_titles[i]
            tickers   = self.ticker_extractor.extract(title, all_texts[i])
            if not tickers: continue

            macro     = self._macro_lookup.get(date_str, {})
            mkt_checks = []

            for ticker in tickers:
                ohlcv, actual_date = self._get_ohlcv(ticker, date_str)
                if ohlcv is None: miss += 1; continue

                open_p  = float(ohlcv.get(col["open"],  0) or 0)
                close_p = float(ohlcv.get(col["close"], 0) or 0)
                vol_v   = float(ohlcv.get(col["volume"],0) or 0)
                avg_vol = float(ohlcv.get(col["avg_vol"],1) or 1)
                if open_p == 0 and close_p == 0: continue

                anomaly   = self.anomaly_detector.classify(sentiment, open_p, close_p, vol_v, avg_vol)
                bdm       = self.bdm_confirmation.confirm(
                    anomaly["anomaly_type"],
                    ohlcv.get(col["net_foreign"], float("nan")) or float("nan"),
                    ohlcv.get(col["net_nonfin"],  float("nan")) or float("nan"),
                )
                sektor    = self.sector_map.get(ticker, "Unknown")
                macro_ctx = self.macro_evaluator.evaluate(
                    sektor,
                    macro.get(col["usdidr"], float("nan")) or float("nan"),
                    macro.get(col["brent"],  float("nan")) or float("nan"),
                    macro.get(col["coal"],   float("nan")) or float("nan"),
                    macro.get(col["bi_rate"], None),
                )

                # ── GROUP ID ───────────────────────────────────────────────
                # Digunakan di training untuk grouped-random split
                # Semua berita BBRI di minggu yang sama → group_id identik
                group_id = self._make_group_id(ticker, actual_date or date_str)

                mkt_checks.append({
                    "ticker":           ticker,
                    "sektor":           sektor,
                    "group_id":         group_id,        # ← BARU
                    "anomaly":          anomaly["anomaly_type"],
                    "price_change_pct": anomaly["price_change_pct"],
                    "volume_ratio":     anomaly["volume_ratio"],
                    "close_vs_open":    round(close_p - open_p, 2),
                    "intraday_return":  float(ohlcv.get(col["intraday_move"], 0) or 0),
                    "daily_return":     float(ohlcv.get(col["daily_return"],  0) or 0),
                    "bdm_status":       bdm["bdm_status"],
                    "bdm_reason":       bdm["bdm_reason"],
                    "macro_tailwind":   macro_ctx["macro_tailwind"],
                    "macro_headwind":   macro_ctx["macro_headwind"],
                })

            if not mkt_checks: continue
            signals.append({
                "date":             date_str,
                "title":            title,
                "sentiment_label":  sentiment,
                "sentiment_score":  score,
                "explicit_tickers": [mc["ticker"] for mc in mkt_checks],
                "market_checks":    mkt_checks,
            })

        # Stats
        anom_counts = {}
        for sig in signals:
            for mc in sig["market_checks"]:
                anom_counts[mc["anomaly"]] = anom_counts.get(mc["anomaly"],0)+1
        total = sum(anom_counts.values())
        logger.info(f"Total signals: {len(signals):,} | Checks: {total:,} | Miss: {miss:,}")
        for k,v in sorted(anom_counts.items(), key=lambda x:-x[1]):
            logger.info(f"  {k:25s}: {v:6,} ({v/total:.1%})")

        if not output_path.endswith(".json"):
            output_path = output_path.rsplit(".",1)[0] + ".json"
        with open(output_path,"w",encoding="utf-8") as f:
            json.dump(signals, f, ensure_ascii=False, indent=2)
        logger.info(f"✅ Saved: {output_path}")
        return signals

print("✅ BEIPipeline v5 ready")


In [ ]:
def load_news_files(macro_path, market_path):
    def read(path):
        with open(path,"r",encoding="utf-8") as f:
            first = f.read(1); f.seek(0)
            return json.load(f) if first=="[" else [json.loads(l) for l in f if l.strip()]
    frames = []
    for path, cat in [(macro_path,"macro"),(market_path,"market")]:
        if not Path(path).exists(): logger.warning(f"Missing: {path}"); continue
        df = pd.DataFrame(read(path))
        df["news_category"] = cat
        frames.append(df)
        logger.info(f"Loaded {len(df):,} dari {Path(path).name}")
    if not frames: return pd.DataFrame()
    comb = pd.concat(frames, ignore_index=True)
    comb["date"] = pd.to_datetime(comb["date"], errors="coerce")
    return comb.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

print("✅ load_news_files ready")


In [ ]:
def build_training_dataset(signals_path, master_path, output_path):
    """
    Bangun training_dataset_v5.parquet dari signals JSON.

    Yang BARU vs versi sebelumnya:
    ─────────────────────────────
    1. group_id tersimpan → dipakai untuk grouped-random split di notebook model
    2. Label: alpha_5d (market-neutral vs IHSG) dengan threshold ±1%
       → purified label: ambiguous zone dibuang, bukan di-assign
    3. Semua data 2020–2026 masuk → split dilakukan di notebook model
    4. Technical features dihitung sekali di sini, bukan di notebook model
    """
    import warnings; warnings.filterwarnings("ignore")

    print("Loading signals...")
    with open(signals_path) as f:
        data = json.load(f)

    df = pd.json_normalize(data, record_path="market_checks",
                           meta=["date","title","sentiment_label","sentiment_score"],
                           errors="ignore")
    df["date"] = pd.to_datetime(df["date"])
    print(f"Signal rows: {df.shape}")

    print("Loading master_df...")
    master = pd.read_parquet(master_path)
    master["date"] = pd.to_datetime(master["date"])
    master = master.sort_values(["ticker","date"])

    # ── Hitung forward returns semua horizon ──────────────────────────────────
    print("Computing forward returns...")
    for h in [1, 3, 5, 10]:
        master[f"fwd_{h}d"] = master.groupby("ticker")["close"].transform(
            lambda x: x.shift(-h) / x - 1
        )

    # ── Hitung technical features ─────────────────────────────────────────────
    print("Computing technical features...")
    grp = master.groupby("ticker")

    master["sma_5"]         = grp["close"].transform(lambda x: x.rolling(5,  min_periods=1).mean())
    master["sma_20"]        = grp["close"].transform(lambda x: x.rolling(20, min_periods=1).mean())
    master["sma_60"]        = grp["close"].transform(lambda x: x.rolling(60, min_periods=1).mean())
    master["price_vs_sma20"]= master["close"] / master["sma_20"] - 1
    master["price_vs_sma60"]= master["close"] / master["sma_60"] - 1

    # RSI-14
    def rsi14(x):
        delta = x.diff()
        gain  = delta.clip(lower=0).rolling(14, min_periods=1).mean()
        loss  = (-delta.clip(upper=0)).rolling(14, min_periods=1).mean().replace(0, 1e-9)
        return 100 - (100 / (1 + gain/loss))
    master["rsi_14"] = grp["close"].transform(rsi14)

    # Volume ratio
    master["avg_vol_20"]  = grp["volume"].transform(lambda x: x.rolling(20,min_periods=1).mean())
    master["vol_ratio"]   = master["volume"] / master["avg_vol_20"].replace(0, 1e-9)

    # Volatility (20-day rolling std of daily return)
    master["volatility_20"] = grp["daily_return"].transform(
        lambda x: x.rolling(20,min_periods=5).std()
    )

    # ── IHSG benchmark untuk alpha ────────────────────────────────────────────
    ihsg = master[master["ticker"]=="IHSG"][["date","close"]].copy().sort_values("date")
    if len(ihsg) > 0:
        for h in [1,3,5,10]:
            ihsg[f"ihsg_fwd_{h}d"] = ihsg["close"].shift(-h) / ihsg["close"] - 1
        ihsg_lookup = ihsg.set_index("date")[[f"ihsg_fwd_{h}d" for h in [1,3,5,10]]].to_dict("index")
        print(f"IHSG rows: {len(ihsg):,}")
    else:
        # Proxy: median return semua saham per hari
        print("IHSG tidak ditemukan, menggunakan market median proxy")
        mkt_median = master.groupby("date")["daily_return"].median()
        for h in [1,3,5,10]:
            mkt_med_fwd = mkt_median.rolling(h).sum().shift(-h)
            master[f"ihsg_fwd_{h}d"] = master["date"].map(mkt_med_fwd)
        ihsg_lookup = None

    # ── Merge ke signal df ────────────────────────────────────────────────────
    TECH_COLS = ["close","sma_5","sma_20","sma_60","rsi_14","vol_ratio",
                 "volatility_20","price_vs_sma20","price_vs_sma60","market_cap",
                 "usdidr","brent_usd","coal_usd","bi_rate_pct",
                 "fwd_1d","fwd_3d","fwd_5d","fwd_10d"]
    avail = [c for c in TECH_COLS if c in master.columns]
    merged = df.merge(
        master[["date","ticker"]+avail].drop_duplicates(["date","ticker"]),
        on=["date","ticker"], how="left"
    )

    # ── Alpha vs IHSG ─────────────────────────────────────────────────────────
    for h in [1,3,5,10]:
        fwd_col = f"fwd_{h}d"
        if fwd_col not in merged.columns: continue
        if ihsg_lookup:
            merged[f"ihsg_fwd_{h}d"] = merged["date"].map(
                lambda d: ihsg_lookup.get(d,{}).get(f"ihsg_fwd_{h}d", np.nan)
            )
        merged[f"alpha_{h}d"] = merged[fwd_col] - merged.get(f"ihsg_fwd_{h}d", pd.Series([0]*len(merged)))

    # ── Purified label (alpha_5d, threshold ±1%) ──────────────────────────────
    ALPHA_COL = "alpha_5d"
    THR       = 0.01
    if ALPHA_COL in merged.columns:
        merged["label"] = np.where(
            merged[ALPHA_COL] > THR,  1,
            np.where(merged[ALPHA_COL] < -THR, 0, np.nan)
        )
    else:
        # Fallback: absolute return
        merged["label"] = (merged.get("fwd_5d", 0) > 0).astype(float)

    before = len(merged)
    merged = merged.dropna(subset=["label","close"]).copy()
    merged["label"] = merged["label"].astype(int)
    print(f"Rows setelah drop ambiguous: {before:,} → {len(merged):,}")
    print(f"Label: {merged['label'].value_counts().to_dict()}")

    # ── group_id sudah ada dari pipeline ─────────────────────────────────────
    if "group_id" not in merged.columns:
        # Fallback jika pipeline lama
        def make_gid(row):
            try:
                dt = pd.Timestamp(row["date"])
                wk = f"{dt.year}W{dt.isocalendar()[1]:02d}"
            except:
                wk = "UNKNOWN"
            return f"{row['ticker']}_{wk}"
        merged["group_id"] = merged.apply(make_gid, axis=1)

    n_groups = merged["group_id"].nunique()
    print(f"Unique groups: {n_groups:,} | Avg rows/group: {len(merged)/n_groups:.1f}")

    merged.to_parquet(output_path, index=False)
    print(f"\n✅ Training dataset saved: {output_path}")
    print(f"   Shape: {merged.shape}")
    print(f"   Columns: {list(merged.columns)}")
    return merged

print("✅ build_training_dataset ready")


In [ ]:
# ═══════════════════════════════════════════════════════
# EXECUTE — Run pipeline dan build training dataset
# ═══════════════════════════════════════════════════════

# Step A: Load data
master_df = pd.read_parquet(MASTER_PATH)
news_df   = load_news_files(MACRO_NEWS, MARKET_NEWS)
try:
    with open(SECTORS_PATH) as f:
        sector_df = pd.DataFrame(json.load(f))
except Exception as e:
    logger.warning(f"sector_df gagal load: {e}"); sector_df = None

# Step B: Run pipeline
pipeline = BEIPipeline(master_df, news_df, sector_df,
                       batch_size=128, max_length=128, neutral_threshold=0.70)
signals  = pipeline.run(output_path=SIGNALS_PATH)
print(f"Signals: {len(signals):,}")

# Step C: Build training dataset (with group_id + purified labels + tech features)
training_df = build_training_dataset(SIGNALS_PATH, MASTER_PATH, DATASET_PATH)
print("\n✅ Pipeline v5 selesai. Lanjut ke 03_pretrained_model_v3.ipynb")


In [ ]:
# Inspeksi hasil
df_check = pd.read_parquet(DATASET_PATH)
print(f"Shape: {df_check.shape}")
print(f"\nKolom: {list(df_check.columns)}")
print(f"\nLabel: {df_check['label'].value_counts().to_dict()}")
print(f"Sektor Known: {(df_check['sektor']!= 'Unknown').mean():.1%}")
print(f"group_id sample: {df_check['group_id'].head(5).tolist()}")
print(f"Unique groups: {df_check['group_id'].nunique():,}")
print(f"\nDate range: {df_check['date'].min().date()} → {df_check['date'].max().date()}")
print(f"\nSample rows:")
print(df_check.sample(3)[["date","ticker","group_id","anomaly","sentiment_label","label","alpha_5d"]].to_string())
